In [30]:
import pandas as pd
import sqlite3

In [31]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

query = """
with user_commits as (
    select t.uid,
    t.first_view_ts,
    t.first_commit_ts,
    datetime(d.deadlines, 'unixepoch') as deadline,
    (julianday(datetime(d.deadlines, 'unixepoch')) - julianday(first_commit_ts)) * 24 as avg_diff
    from test t
    join deadlines d on t.labname = d.labs
    where t.labname != 'project1'
),
before_after as (
    select uid
    from user_commits
    where first_commit_ts < first_view_ts
    intersect
    select uid
    from user_commits
    where first_commit_ts >= first_view_ts
)
select 'before' as time,
avg(uc.avg_diff) as avg_diff
from user_commits uc
where uc.uid in (select * from before_after)
and uc.first_commit_ts < uc.first_view_ts
union all
select 'after' as time,
avg(uc.avg_diff) as avg_diff
from user_commits uc
where uc.uid in (select uid from before_after)
and uc.first_commit_ts >= uc.first_view_ts
"""

test_results = pd.read_sql(query, conn)
test_results

,time,avg_diff
0,before,61.156438
1,after,105.229101


In [32]:
query = """
with user_commits as (
    select c.uid,
    c.first_view_ts,
    c.first_commit_ts,
    datetime(d.deadlines, 'unixepoch') as deadline,
    (julianday(datetime(d.deadlines, 'unixepoch')) - julianday(first_commit_ts)) * 24 as avg_diff
    from control c
    join deadlines d on c.labname = d.labs
    where c.labname != 'project1'
),
before_after as (
    select uid
    from user_commits
    where first_commit_ts < first_view_ts
    intersect
    select uid
    from user_commits
    where first_commit_ts >= first_view_ts
)
select 'before' as time,
avg(uc.avg_diff) as avg_diff
from user_commits uc
where uc.uid in (select * from before_after)
and uc.first_commit_ts < uc.first_view_ts
union all
select 'after' as time,
avg(uc.avg_diff) as avg_diff
from user_commits uc
where uc.uid in (select uid from before_after)
and uc.first_commit_ts >= uc.first_view_ts
"""

control_results = pd.read_sql(query, conn)
control_results

,time,avg_diff
0,before,99.901295
1,after,118.144425


In [33]:
conn.close

<function Connection.close>

Ответ: Да, гипотеза подтвердилась.